In [57]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline

from xgboost import XGBClassifier

In [58]:
import pandas as pd

columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
    'income'
]

train = pd.read_csv(
    '../data/adult.data',
    header=None,
    names=columns,
    skipinitialspace=True
)

test = pd.read_csv(
    '../data/adult.test',
    header=None,
    names=columns,
    skipinitialspace=True,
    skiprows=1
)

print(train.shape)

df = pd.concat([train, test], ignore_index=True)

df.head()

(32561, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [59]:
df['income'] = df['income'].replace({
    '<=50K.': '<=50K',
    '>50K.': '>50K'
})

print(df['income'].value_counts())

income
<=50K    37155
>50K     11687
Name: count, dtype: int64


In [60]:
df = df.drop_duplicates()

print("Duplicate rows:", df.duplicated().sum())
print("Dataset shape:", df.shape)

Duplicate rows: 0
Dataset shape: (48790, 15)


In [61]:
import numpy as np

df = df.replace('?', np.nan)

print("Missing values:")
print(df.isnull().sum())

Missing values:
age                  0
workclass         2795
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        2805
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     856
income               0
dtype: int64


In [62]:
df['capital-net'] = df['capital-gain'] - df['capital-loss']

df = df.drop(
    ['capital-gain', 'capital-loss'],axis=1
)

df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,hours-per-week,native-country,income,capital-net
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,40,United-States,<=50K,2174
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,13,United-States,<=50K,0
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,40,United-States,<=50K,0
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,40,United-States,<=50K,0
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,40,Cuba,<=50K,0


In [63]:
# Split data into train and test
# Using iloc as done in the original project

train = df.iloc[:32561].copy()
test = df.iloc[32561:].copy()

print("Train shape:", train.shape)
print("Test shape :", test.shape)

Train shape: (32561, 14)
Test shape : (16229, 14)


In [64]:
X_train = train.drop('income', axis=1)
y_train = train['income']

X_test = test.drop('income', axis=1)
y_test = test['income']

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

X_train: (32561, 13)
X_test : (16229, 13)


In [65]:
fnlwgt_median = X_train["fnlwgt"].median()
print(f"Training median fnlwgt: {fnlwgt_median}")

Training median fnlwgt: 178356.0


In [48]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='most_frequent')

cols = ['workclass', 'occupation', 'native-country']

X_train[cols] = imputer.fit_transform(X_train[cols])
X_test[cols] = imputer.transform(X_test[cols])

print("Imputation completed.")

Imputation completed.


In [49]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

print("Classes:", label_encoder.classes_)

Classes: ['<=50K' '>50K']


In [50]:
# Numerical and categorical columns


categorical_cols = [
    'workclass',
    'education',
    'marital-status',
    'occupation',
    'relationship',
    'race',
    'native-country'
]

# Sex is handled separately using OrdinalEncoder
categorical_indices = [
    X_train.columns.get_loc(col)
    for col in categorical_cols + ['sex']
]

print("Categorical columns:", categorical_cols)
print("Categorical indices:", categorical_indices)


Categorical columns: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'native-country']
Categorical indices: [1, 3, 5, 6, 7, 8, 11, 9]


In [51]:

smote = SMOTENC(
    categorical_features=categorical_indices,
    random_state=42
)


categorical_transformer = OneHotEncoder(
    handle_unknown='ignore'
)

sex_transformer = OrdinalEncoder()

In [52]:

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_cols),
        ("cat", OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ("sex", OrdinalEncoder(), ["sex"])
    ]
)

In [53]:
# Best Parameters we got from hyperparameter tuning:
# {
#     'Model__colsample_bytree': 1.0,
#     'Model__learning_rate': 0.2,
#     'Model__max_depth': 4,
#     'Model__n_estimators': 200,
#     'Model__subsample': 1.0
# }

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.2,
    max_depth=4,
    subsample=1.0,
    colsample_bytree=1.0,
    random_state=42,
    objective='binary:logistic',
    eval_metric='logloss'
)

In [54]:
from imblearn.pipeline import Pipeline

xgb_pipeline = Pipeline(
    steps=[
        ("SMOTE", smote),
        ("Preprocessor", tree_preprocessor),
        ("Model", xgb_model)
    ]
)

In [55]:
# Fit the final tuned XGBoost pipeline
xgb_pipeline.fit(X_train, y_train)

print("Final XGBoost pipeline fitted successfully.")

Final XGBoost pipeline fitted successfully.


In [56]:
import os
import joblib

os.makedirs("model", exist_ok=True)

joblib.dump(xgb_pipeline, "model/final_xgb_model.pkl")
joblib.dump(imputer, "model/imputer.pkl")
joblib.dump(label_encoder, "model/label_encoder.pkl")

print("All deployment files saved successfully.")

All deployment files saved successfully.
